## **2026.02.18_01**

**CELL 1: Imports & Setup**
```python
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print("="*120)
print("DISCOVERING RELATIONSHIP: y = f(x) using XGBoost")
print("="*120)
```

---

**CELL 2: Data Preparation**
```python
print("\n[STEP 1] Preparing Data...")

product_tags = [
    'FIC10014_PV', 'FIC20017_PV', 'FIC20018_PV', 'FIC25021_PV', 
    'FIC30018_PV', 'FIC40039_PV', 'FI40039_PV', 'FIC40045_PV', 
    'FIC50011_PV', 'FIC60020_PV', 'FIC65004_PV', 'FIC70053_PV', 'FIC95008_PV'
]

utility_tags = [tag for tag in up_wide.columns if tag not in product_tags]
product_tags = [t for t in product_tags if t in up_wide.columns]
utility_tags = [t for t in utility_tags if t in up_wide.columns]

print(f"  Utilities (Input): {len(utility_tags)}")
print(f"  Products (Output): {len(product_tags)}")

# Extract data
X = up_wide[utility_tags].values
y = up_wide[product_tags].values

# Remove NaN
valid_idx = (~np.isnan(X).any(axis=1)) & (~np.isnan(y).any(axis=1))
X = X[valid_idx]
y = y[valid_idx]

print(f"  Data shape: {X.shape}")

# Normalize
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42
)

print(f"  Train: {X_train.shape[0]}, Test: {X_test.shape[0]} ✓")
```

---

**CELL 3: Train XGBoost Models**
```python
print("\n[STEP 2] Training XGBoost Models for Each Product...")
print("="*120)

models = {}
metrics_list = []

for prod_idx, prod_tag in enumerate(product_tags):
    print(f"Training {prod_tag}...", end=" ")
    
    # Train XGBoost
    model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
    
    model.fit(X_train, y_train[:, prod_idx], verbose=False)
    
    # Predictions
    y_pred_test = model.predict(X_test)
    y_test_prod = y_test[:, prod_idx]
    
    # Metrics
    r2 = r2_score(y_test_prod, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test_prod, y_pred_test))
    mae = mean_absolute_error(y_test_prod, y_pred_test)
    
    quality = '⭐ Excellent' if r2 > 0.85 else '✓ Good' if r2 > 0.75 else '⚠ Fair'
    
    models[prod_tag] = model
    metrics_list.append({
        'Product': prod_tag,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Quality': quality
    })
    
    print(f"R²: {r2:.4f} {quality}")

print("="*120)
print("Training Complete! ✓")
```

---

**CELL 4: Display Metrics**
```python
print("\n[STEP 3] Model Evaluation Metrics...")
print("="*120)

metrics_df = pd.DataFrame(metrics_list)

print("\nPER-PRODUCT METRICS:")
for _, row in metrics_df.iterrows():
    print(f"{row['Product']:20} | R²: {row['R²']:7.4f} | RMSE: {row['RMSE']:8.4f} | {row['Quality']}")

overall_r2 = metrics_df['R²'].mean()
print(f"\nOverall R² Average: {overall_r2:.4f}")
print("="*120)

display(metrics_df)
```

---

**CELL 5: Feature Importance (Discover Relationships)**
```python
print("\n[STEP 4] Extracting Relationships (Feature Importance)...")
print("="*120)

relationship_dict = {}

for prod_tag, model in models.items():
    # Get feature importance
    importance = model.feature_importances_
    
    # Create dataframe
    importance_df = pd.DataFrame({
        'Utility': utility_tags,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    relationship_dict[prod_tag] = importance_df
    
    print(f"\n{prod_tag}:")
    for _, row in importance_df.head(5).iterrows():
        print(f"  {row['Utility']:20} → {row['Importance']:.4f}")

print("\n" + "="*120)
```

---

**CELL 6: Create Relationship Matrix**
```python
print("\n[STEP 5] Creating Relationship Matrix...")

# Create matrix
relationship_matrix = pd.DataFrame(
    index=product_tags,
    columns=utility_tags,
    data=0.0
)

for prod_tag, importance_df in relationship_dict.items():
    for _, row in importance_df.iterrows():
        relationship_matrix.loc[prod_tag, row['Utility']] = row['Importance']

print("\nRelationship Matrix (Utility Importance for Each Product):")
display(relationship_matrix)

# Save
relationship_matrix.to_csv('xgboost_relationships.csv')
print("\n✓ Saved: xgboost_relationships.csv")
```

---

**CELL 7: Visualize Relationships Heatmap**
```python
print("\n[STEP 6] Visualizing Relationships...")

fig = go.Figure(data=go.Heatmap(
    z=relationship_matrix.values.astype(float),
    x=relationship_matrix.columns,
    y=relationship_matrix.index,
    colorscale='Viridis',
    text=np.round(relationship_matrix.values.astype(float), 3),
    texttemplate='%{text:.3f}',
    textfont={"size": 8},
    colorbar=dict(title='Importance')
))

fig.update_layout(
    title='Utility → Product Relationships (XGBoost Feature Importance)',
    xaxis_title='Utilities (Inputs)',
    yaxis_title='Products (Outputs)',
    height=700,
    width=1200,
    xaxis=dict(tickangle=45)
)

fig.show()
```

---

**CELL 8: Top Utilities for Each Product**
```python
print("\n[STEP 7] Top 5 Utilities for Each Product...")
print("="*120)

for prod_tag in product_tags:
    top_utils = relationship_dict[prod_tag].head(5)
    print(f"\n{prod_tag}:")
    for _, row in top_utils.iterrows():
        bar = "█" * int(row['Importance'] * 50)
        print(f"  {row['Utility']:20} {bar} {row['Importance']:.4f}")
```

---

**CELL 9: Make Predictions (y = f(x))**
```python
print("\n[STEP 8] Making Predictions on Full Dataset...")

# Predict all products
y_pred_all = np.zeros_like(y_norm)

for prod_idx, (prod_tag, model) in enumerate(models.items()):
    y_pred_all[:, prod_idx] = model.predict(X_norm)

# Denormalize
y_pred = scaler_y.inverse_transform(y_pred_all)
y_actual = scaler_y.inverse_transform(y_norm)

print(f"✓ Predictions made for {len(product_tags)} products")
print(f"  Prediction shape: {y_pred.shape}")
```

---

**CELL 10: Calculate Energy Efficiency**
```python
print("\n[STEP 9] Extracting Energy Efficiency...")

# Energy Efficiency = Actual / Predicted
efficiency = (y_actual.sum(axis=1) / (y_pred.sum(axis=1) + 1e-10)) * 100

print(f"\nEnergy Efficiency Statistics:")
print(f"  Mean: {efficiency.mean():.2f}%")
print(f"  Min: {efficiency.min():.2f}%")
print(f"  Max: {efficiency.max():.2f}%")
print(f"  Std: {efficiency.std():.2f}%")

# Create efficiency dataframe
efficiency_df = pd.DataFrame({
    'Efficiency_%': efficiency,
    'Actual_Output': y_actual.sum(axis=1),
    'Predicted_Output': y_pred.sum(axis=1)
})

display(efficiency_df.head())
```

---

**CELL 11: Visualize Efficiency Over Time**
```python
fig = go.Figure()

fig.add_trace(go.Scatter(
    y=efficiency,
    mode='lines',
    name='Energy Efficiency',
    line=dict(color='green', width=2)
))

fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Baseline (100%)")

fig.update_layout(
    title='Energy Efficiency Over Time',
    xaxis_title='Sample',
    yaxis_title='Efficiency (%)',
    height=500,
    width=1200,
    hovermode='x unified'
)

fig.show()
```

---

**CELL 12: Summary**
```python
print("\n" + "="*120)
print("MODEL SUMMARY: y = f(x) using XGBoost")
print("="*120)

summary = f"""
DISCOVERED RELATIONSHIP:

Input (x):  {len(utility_tags)} Utilities
Output (y): {len(product_tags)} Products

Model: XGBoost (Gradient Boosting)
  • 100 estimators, max_depth=6
  • Trained on {X_train.shape[0]} samples
  • Tested on {X_test.shape[0]} samples

Performance:
  • Average R²: {overall_r2:.4f}
  • Products with R² > 0.85: {(metrics_df['R²'] > 0.85).sum()}/{len(product_tags)}

What You Can Do:
  ✓ Predict products from utilities
  ✓ Measure efficiency (actual/predicted)
  ✓ Detect anomalies
  ✓ See which utilities matter most
  ✓ Optimize plant operations

Files Saved:
  • xgboost_relationships.csv
"""

print(summary)

# Save efficiency column
efficiency_df.to_csv('energy_efficiency.csv', index=False)
metrics_df.to_csv('xgboost_metrics.csv', index=False)
print("✓ All results saved!")
```

---

Done! Copy these 12 cells into Jupyter! 🎯

In [ ]:
## **2026.02.14_01**

## **2026.02.14_01**

```python
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy import stats

print("="*80)
print("PRODUCT OPTIMAL RANGES WITH SUPPORTING UTILITY RANGES")
print("="*80)

# Define product tags
product_tags = [
    'FIC10014_PV', 'FIC20017_PV', 'FIC20018_PV', 'FIC25021_PV', 
    'FIC30018_PV', 'FIC40039_PV', 'FI40039_PV', 'FIC40045_PV', 
    'FIC50011_PV', 'FIC60020_PV', 'FIC65004_PV', 'FIC70053_PV', 'FIC95008_PV'
]

# Get utility tags
utility_tags = [tag for tag in up_wide.columns if tag not in product_tags]

# ============================================================================
# 1. EXTRACT OPTIMAL RANGES FOR PRODUCTS
# ============================================================================

optimal_ranges = pd.DataFrame()

for tag in product_tags:
    if tag not in up_wide.columns:
        continue
    
    data = up_wide[tag].dropna()
    Q1, Q3 = data.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    clean_data = data[(data >= Q1 - 1.5*IQR) & (data <= Q3 + 1.5*IQR)]
    
    optimal_min = clean_data.quantile(0.25)
    optimal_max = clean_data.quantile(0.75)
    
    efficiency = clean_data.mean() / (clean_data.std() + 1e-10)
    cv = clean_data.std() / clean_data.mean()
    stability = 1 / (1 + cv)
    reliability = len(clean_data) / len(data)
    score = (efficiency + stability + reliability) / 3
    
    optimal_ranges = pd.concat([optimal_ranges, pd.DataFrame({
        'Product_Tag': [tag],
        'Optimal_Min': [optimal_min],
        'Optimal_Max': [optimal_max],
        'Optimal_Center': [clean_data.mean()],
        'Score': [score],
        'Stability': [stability],
        'Reliability': [reliability]
    })], ignore_index=True)

# ============================================================================
# 2. FOR EACH PRODUCT, FIND UTILITY RANGES WHEN PRODUCT IS OPTIMAL
# ============================================================================

print("\nCALCULATING SUPPORTING UTILITY RANGES...")
print("-" * 80)

final_table = []

for _, prod_row in optimal_ranges.iterrows():
    prod_tag = prod_row['Product_Tag']
    prod_opt_min = prod_row['Optimal_Min']
    prod_opt_max = prod_row['Optimal_Max']
    
    # Find indices when product is in optimal range
    prod_data = up_wide[prod_tag]
    optimal_mask = (prod_data >= prod_opt_min) & (prod_data <= prod_opt_max)
    optimal_indices = prod_data[optimal_mask].index
    
    print(f"\n{prod_tag}:")
    print(f"  Optimal Range: {prod_opt_min:.0f} - {prod_opt_max:.0f}")
    print(f"  Points in optimal range: {optimal_mask.sum()} / {len(prod_data)}")
    
    # Calculate correlations with utilities when product is optimal
    utility_support = {}
    for util_tag in utility_tags:
        if util_tag in up_wide.columns:
            util_data = up_wide.loc[optimal_indices, util_tag]
            prod_data_opt = up_wide.loc[optimal_indices, prod_tag]
            
            # Remove NaN
            valid_mask = (~util_data.isna()) & (~prod_data_opt.isna())
            
            if valid_mask.sum() > 5:  # Need at least 5 points
                corr = util_data[valid_mask].corr(prod_data_opt[valid_mask])
                
                if abs(corr) > 0.2:  # Only keep significant correlations
                    # Get utility range when product is optimal
                    util_range_min = util_data[valid_mask].quantile(0.25)
                    util_range_max = util_data[valid_mask].quantile(0.75)
                    util_mean = util_data[valid_mask].mean()
                    
                    utility_support[util_tag] = {
                        'correlation': corr,
                        'min': util_range_min,
                        'max': util_range_max,
                        'mean': util_mean,
                        'count': valid_mask.sum()
                    }
    
    # Sort by absolute correlation
    sorted_utils = sorted(utility_support.items(), 
                         key=lambda x: abs(x[1]['correlation']), 
                         reverse=True)
    
    # Get top 3 utilities
    row_data = {
        'Product_Tag': prod_tag,
        'Prod_Opt_Min': prod_opt_min,
        'Prod_Opt_Max': prod_opt_max,
        'Prod_Score': prod_row['Score'],
        'Prod_Stability': prod_row['Stability'],
        'Prod_Reliability': prod_row['Reliability']
    }
    
    for idx, (util_tag, util_info) in enumerate(sorted_utils[:3]):
        prefix = f'Utility{idx+1}'
        row_data[f'{prefix}_Tag'] = util_tag
        row_data[f'{prefix}_Corr'] = util_info['correlation']
        row_data[f'{prefix}_Min'] = util_info['min']
        row_data[f'{prefix}_Max'] = util_info['max']
        row_data[f'{prefix}_Mean'] = util_info['mean']
    
    # Fill empty utility columns if less than 3
    for idx in range(len(sorted_utils) + 1, 4):
        prefix = f'Utility{idx}'
        row_data[f'{prefix}_Tag'] = 'N/A'
        row_data[f'{prefix}_Corr'] = 0
        row_data[f'{prefix}_Min'] = 0
        row_data[f'{prefix}_Max'] = 0
        row_data[f'{prefix}_Mean'] = 0
    
    final_table.append(row_data)

# Create final DataFrame
final_df = pd.DataFrame(final_table)

# ============================================================================
# 3. DISPLAY MAIN TABLE
# ============================================================================

print("\n" + "="*80)
print("PRODUCT OPTIMIZATION TABLE WITH SUPPORTING UTILITY RANGES")
print("="*80)

# Create simplified display table
display_table = pd.DataFrame()

for _, row in final_df.iterrows():
    display_table = pd.concat([display_table, pd.DataFrame({
        'Product': [row['Product_Tag']],
        'Prod_Range': [f"{row['Prod_Opt_Min']:.0f}-{row['Prod_Opt_Max']:.0f}"],
        'Score': [f"{row['Prod_Score']:.3f}"],
        'Primary_Utility': [row['Utility1_Tag']],
        'Util1_Range': [f"{row['Utility1_Min']:.0f}-{row['Utility1_Max']:.0f}"],
        'Util1_Corr': [f"{row['Utility1_Corr']:.3f}"],
        'Secondary_Utility': [row['Utility2_Tag']],
        'Util2_Range': [f"{row['Utility2_Min']:.0f}-{row['Utility2_Max']:.0f}"],
        'Util2_Corr': [f"{row['Utility2_Corr']:.3f}"],
        'Tertiary_Utility': [row['Utility3_Tag']],
        'Util3_Range': [f"{row['Utility3_Min']:.0f}-{row['Utility3_Max']:.0f}"],
        'Util3_Corr': [f"{row['Utility3_Corr']:.3f}"]
    })], ignore_index=True)

print("\n" + display_table.to_string(index=False))

# ============================================================================
# 4. SAVE DETAILED TABLE
# ============================================================================

print("\n" + "="*80)
print("DETAILED OPTIMIZATION REFERENCE TABLE")
print("="*80)

detailed_table = []

for _, row in final_df.iterrows():
    print(f"\n{'='*80}")
    print(f"PRODUCT: {row['Product_Tag']}")
    print(f"{'='*80}")
    print(f"Optimal Product Range: {row['Prod_Opt_Min']:.0f} - {row['Prod_Opt_Max']:.0f}")
    print(f"Optimization Score: {row['Prod_Score']:.3f} (Stability: {row['Prod_Stability']:.3f}, Reliability: {row['Prod_Reliability']:.3f})")
    print()
    
    for util_idx in range(1, 4):
        util_tag = row[f'Utility{util_idx}_Tag']
        if util_tag != 'N/A':
            corr = row[f'Utility{util_idx}_Corr']
            min_val = row[f'Utility{util_idx}_Min']
            max_val = row[f'Utility{util_idx}_Max']
            mean_val = row[f'Utility{util_idx}_Mean']
            
            print(f"  Utility #{util_idx}: {util_tag}")
            print(f"    └─ Correlation: {corr:.3f}")
            print(f"    └─ Optimal Range: {min_val:.0f} - {max_val:.0f}")
            print(f"    └─ Target Mean: {mean_val:.0f}")
            print()

# ============================================================================
# 5. CREATE VISUALIZATION: PRODUCT vs UTILITIES
# ============================================================================

print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

# Create subplots for top 4 products
top_products = final_df.nlargest(4, 'Prod_Score')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{row['Product_Tag']}\n(Score: {row['Prod_Score']:.3f})" 
                    for _, row in top_products.iterrows()],
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

for idx, (_, prod_row) in enumerate(top_products.iterrows()):
    row_num = idx // 2 + 1
    col_num = idx % 2 + 1
    
    prod_tag = prod_row['Product_Tag']
    
    # Get utilities and correlations
    util_names = []
    util_corrs = []
    util_ranges = []
    
    for util_idx in range(1, 4):
        util_tag = prod_row[f'Utility{util_idx}_Tag']
        if util_tag != 'N/A':
            util_names.append(util_tag)
            util_corrs.append(prod_row[f'Utility{util_idx}_Corr'])
            min_val = prod_row[f'Utility{util_idx}_Min']
            max_val = prod_row[f'Utility{util_idx}_Max']
            util_ranges.append(f"{min_val:.0f}-{max_val:.0f}")
    
    text_labels = [f"{name}<br>{r}<br>Corr:{c:.3f}" 
                   for name, r, c in zip(util_names, util_ranges, util_corrs)]
    
    fig.add_trace(
        go.Bar(x=util_names, y=util_corrs, name=prod_tag,
               text=util_ranges, textposition='auto'),
        row=row_num, col=col_num
    )

fig.update_layout(height=800, title_text="Top Products with Supporting Utilities", showlegend=False)
fig.show()

# ============================================================================
# 6. EXPORT TO CSV
# ============================================================================

export_df = final_df[[
    'Product_Tag', 'Prod_Opt_Min', 'Prod_Opt_Max', 'Prod_Score',
    'Utility1_Tag', 'Utility1_Min', 'Utility1_Max', 'Utility1_Corr',
    'Utility2_Tag', 'Utility2_Min', 'Utility2_Max', 'Utility2_Corr',
    'Utility3_Tag', 'Utility3_Min', 'Utility3_Max', 'Utility3_Corr'
]]

export_df.columns = [
    'Product', 'Prod_Min', 'Prod_Max', 'Score',
    'Util1', 'Util1_Min', 'Util1_Max', 'Util1_Corr',
    'Util2', 'Util2_Min', 'Util2_Max', 'Util2_Corr',
    'Util3', 'Util3_Min', 'Util3_Max', 'Util3_Corr'
]

print("\n" + "="*80)
print("EXPORTING TABLE TO CSV")
print("="*80)
print("\nOptimization table ready for export:")
print(export_df.to_string(index=False))

# Save if needed
# export_df.to_csv('product_optimization_with_utilities.csv', index=False)
# print("\nSaved to: product_optimization_with_utilities.csv")

print("\n" + "="*80)
print("SUMMARY RECOMMENDATIONS")
print("="*80)

summary = """
How to Use This Table:

1. FOR EACH PRODUCT TAG:
   └─ See the optimal range where the product performs best
   
2. TO OPTIMIZE THAT PRODUCT:
   └─ Keep the supporting utilities in their specified ranges
   └─ Example: To optimize FIC10014, keep primary utility in its range
   
3. CORRELATION STRENGTH:
   └─ High correlation (0.7+) = critical utility
   └─ Medium correlation (0.4-0.7) = important utility
   └─ Low correlation (0.2-0.4) = supporting utility

4. PRACTICAL APPLICATION:
   └─ Set alarms at utility range boundaries
   └─ When utility drifts out of range, expect product to drift
   └─ Use as baseline for predictive maintenance

5. OPTIMIZATION STRATEGY:
   └─ Maintain primary utility in range first (highest impact)
   └─ Then adjust secondary utilities
   └─ Monitor tertiary utilities for fine-tuning
"""

print(summary)

```

# **2026.02.12_02** 

**Long format** vs **Wide format** - different ways to structure data:

**Wide Format** (what you have now):
```
Datetime          | FI10203_PV | FI20203_PV | FI25205_PV | ...
2021-05-01 01:00 |    333.3   |    378.9   |    749.1   |
2021-05-01 02:00 |    330.6   |    375.8   |    707.8   |
```
- Each TAG is a **column**
- Compact, good for correlation & PCA
- Hard to filter by single TAG

**Long Format** (what you need):
```
Datetime          | TAG        | Value
2021-05-01 01:00 | FI10203_PV |  333.3
2021-05-01 01:00 | FI20203_PV |  378.9
2021-05-01 01:00 | FI25205_PV |  749.1
2021-05-01 02:00 | FI10203_PV |  330.6
2021-05-01 02:00 | FI20203_PV |  375.8
```
- Each TAG is a **row** (with Datetime and Value)
- More rows, easier to filter
- Good for plotting individual TAGs

**Conversion:**

Wide → Long:
```python
long = wide.reset_index().melt(id_vars=['Datetime'], var_name='TAG', value_name='Value')
```

Long → Wide:
```python
wide = long.pivot_table(index='Datetime', columns='TAG', values='Value')
```

**When to use:**
- **Wide:** Correlation, clustering, PCA, multi-variable analysis
- **Long:** Plotting single variables, filtering by TAG, time series of one TAG

Your case needs **long** for the interactive dropdown plot.

---

---

## My Opinion

### Part 1: Feature Readiness Table ⭐⭐⭐⭐⭐

**Highly Recommended** — This is exactly what industry expects.

| Pros | Cons |
|------|------|
| Turns EDA into decisions | Requires domain knowledge for thresholds |
| Reproducible & auditable | Initial setup time |
| Reviewer-friendly | — |

**Verdict:** Implement this. It makes your notebook production-ready.

---

### Part 2: Advanced Time-Series Analysis ⭐⭐⭐⭐

**Good but selective** — Use what's relevant.

| Analysis | Priority | Your Data |
|----------|----------|-----------|
| ADF (Stationarity) | ✅ Must have | Essential for LSTM |
| ACF (Autocorrelation) | ✅ Must have | Helps choose lag/window |
| STL (Decomposition) | ⚠️ Optional | If seasonality expected |
| PCA | ✅ Already done | ✅ |
| t-SNE | ❌ Skip | UMAP is better |
| UMAP | ✅ Recommended | Better than PaCMAP for interpretation |
| Mutual Information | ⚠️ Optional | Good for feature selection |
| VIF | ✅ Recommended | Check multicollinearity |
| ydata_profiling | ⚠️ Appendix only | Heavy, not core |

---

## My Recommendation

**Add these to your notebook:**

### 1. Feature Readiness Table (Part 1) — Essential
```python
from scipy.stats import skew
from statsmodels.tsa.stattools import adfuller

rows = []
for col in pivot.columns:
    s = pivot[col].dropna()
    if len(s) < 100:
        continue
    
    try:
        pval = adfuller(s, autolag="AIC")[1]
        stationary = pval < 0.05
    except:
        stationary = False
    
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_pct = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).mean() * 100
    
    rows.append({
        "feature": col,
        "missing_%": round(pivot[col].isna().mean() * 100, 2),
        "variance": round(s.var(), 2),
        "skewness": round(skew(s), 2),
        "outlier_%": round(outlier_pct, 2),
        "stationary": stationary
    })

feature_stats = pd.DataFrame(rows)

# Decision rules
def feature_decision(row):
    if row["missing_%"] > 40:
        return "DROP", "Too many missing"
    if row["variance"] < 1e-6:
        return "DROP", "Constant signal"
    if abs(row["skewness"]) > 2:
        return "KEEP", "Log transform needed"
    if not row["stationary"]:
        return "KEEP", "Differencing needed"
    return "KEEP", "Stable signal"

def recommend_transform(row):
    if row["decision"] == "DROP":
        return "—"
    if abs(row["skewness"]) > 2:
        return "Log + RobustScaler"
    if not row["stationary"]:
        return "Diff + StandardScaler"
    return "StandardScaler"

decisions = feature_stats.apply(feature_decision, axis=1, result_type="expand")
feature_stats["decision"] = decisions[0]
feature_stats["reason"] = decisions[1]
feature_stats["transformation"] = feature_stats.apply(recommend_transform, axis=1)

display(feature_stats)
```

---

### 2. VIF (Multicollinearity) — Recommended
```python
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

X = pivot.fillna(method="ffill").dropna()
X_scaled = StandardScaler().fit_transform(X)
X_vif = pd.DataFrame(X_scaled, columns=X.columns)

vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

display(vif.sort_values("VIF", ascending=False))
```

**VIF Interpretation:**
| VIF | Meaning |
|-----|---------|
| 1 | No correlation |
| 1-5 | Moderate |
| 5-10 | High → consider removing |
| >10 | Severe → must remove or combine |

---

### 3. ACF (Autocorrelation) — For LSTM window selection
```python
from statsmodels.graphics.tsaplots import plot_acf

tag = pivot.columns[0]
fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(pivot[tag].dropna(), lags=72, ax=ax)
plt.title(f'{tag} - Autocorrelation (helps choose LSTM window)')
plt.show()
```

**Reading ACF:**
- High correlation at lag 24 → daily pattern
- Significant until lag 48 → use 48-hour window for LSTM

---

### 4. UMAP (Better than PaCMAP) — Optional upgrade
```python
!pip install umap-learn -q

import umap

X = pivot.fillna(method="ffill").dropna()
X_scaled = StandardScaler().fit_transform(X)

# PCA first (required)
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)

# Then UMAP
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_pca)

fig = px.scatter(
    x=X_umap[:, 0], y=X_umap[:, 1],
    color=pd.to_datetime(pivot.index).strftime('%Y-%m'),
    title='UMAP: Process States Over Time'
)
fig.show()
```

---

## Final Notebook Structure

```
01_data_exploration.ipynb
├── 1. Data Loading
├── 2. Data Overview (shape, dtypes, date range)
├── 3. Missing Values & Gaps
├── 4. Distribution Analysis (histograms, outliers)
├── 5. Correlation Analysis (Pearson, Spearman, Kendall)
├── 6. Clustering (Dendrogram, PCA on tags)
├── 7. Dimensionality Reduction (PCA, UMAP)
├── 8. Time-Series Analysis (ADF, ACF) ← ADD
├── 9. Multicollinearity (VIF) ← ADD
├── 10. Feature Readiness Table ← ADD
└── 11. Summary & Next Steps
```

This structure is **publication-quality** and **industry-standard**.
```

---

Want me to help you implement any of these?

# **2026.02.12**

Here's a comprehensive structure for all three analyses:

```python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. PREPARE DATA
up['Datetime'] = pd.to_datetime(up['Datetime'])
up = up.sort_values('Datetime')

# Pivot to wide format (each TAG is a column)
up_wide = up.pivot_table(
    index='Datetime',
    columns='TAG',
    values='Value',
    aggfunc='first'
)

# Handle missing values
up_wide = up_wide.fillna(method='ffill').fillna(method='bfill')

# Store metadata separately
metadata = up[['TAG', 'Equipment', 'Desc.', 'Unit', 'SECTION']].drop_duplicates().set_index('TAG')

# 2. FEATURE ENGINEERING for all analyses
df = up_wide.copy()

# Time-based features
df['Hour'] = df.index.hour
df['DayOfWeek'] = df.index.dayofweek
df['Month'] = df.index.month
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Lag features (for forecasting & anomaly detection)
for lag in [1, 24, 168]:  # 1 hour, 1 day, 1 week
    for col in up_wide.columns:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Rolling statistics (for anomaly detection)
for col in up_wide.columns:
    df[f'{col}_rolling_mean'] = df[col].rolling(window=24).mean()
    df[f'{col}_rolling_std'] = df[col].rolling(window=24).std()
    df[f'{col}_rolling_min'] = df[col].rolling(window=24).min()
    df[f'{col}_rolling_max'] = df[col].rolling(window=24).max()

# Differencing (for trend analysis)
for col in up_wide.columns:
    df[f'{col}_diff'] = df[col].diff()

# Drop NaN rows from lag/rolling features
df = df.dropna()

# 3. PREPARE DATASETS FOR EACH ANALYSIS

# ===== FORECASTING =====
# Separate features and target
forecast_features = df.drop(up_wide.columns, axis=1)  # All engineered features
forecast_targets = df[up_wide.columns]  # Original TAG values

# 4. NORMALIZE for all models
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns,
    index=df.index
)

# ===== ANOMALY DETECTION =====
# Use original values + rolling statistics
anomaly_features = df[[col for col in df.columns if 'rolling' in col or col in up_wide.columns]]

# ===== CLUSTERING & CORRELATION =====
# Use original values only (wide format)
clustering_data = df[up_wide.columns]

# 4. SUMMARY STRUCTURE
data_dict = {
    'raw': up,                           # Original data
    'wide': up_wide,                     # Pivoted format
    'engineered': df,                    # With all features
    'scaled': df_scaled,                 # Standardized
    'metadata': metadata,                # TAG information
    
    # For specific analyses:
    'forecasting': {
        'features': forecast_features,
        'targets': forecast_targets,
        'scaled': scaler.transform(forecast_features)
    },
    'anomaly_detection': {
        'features': anomaly_features,
        'scaled': df_scaled[[col for col in df_scaled.columns if 'rolling' in col]]
    },
    'clustering': {
        'data': clustering_data,
        'scaled': df_scaled[up_wide.columns],
        'correlation': clustering_data.corr()
    }
}

# 5. QUICK VERIFICATION
print("Data shapes:")
print(f"Raw: {up.shape}")
print(f"Wide: {up_wide.shape}")
print(f"Engineered: {df.shape}")
print(f"\nAvailable TAGs: {list(up_wide.columns)}")
print(f"\nEngineered features: {list(df.columns)}")
```

**Now you can easily run analyses:**

```python
# FORECASTING
from sklearn.ensemble import RandomForestRegressor
X = data_dict['forecasting']['features']
y = data_dict['forecasting']['targets']

# ANOMALY DETECTION
anomaly_data = data_dict['anomaly_detection']['features']
# Detect outliers using rolling statistics

# CLUSTERING & CORRELATION
corr_matrix = data_dict['clustering']['correlation']
# Run clustering on scaled data
```

This structure keeps everything organized for all three analyses!